# MCI Notebook for Custom Dual Clouds

This notebook computes the MCI of a pair of point sets (a *dual cloud*) that you bring yourself, and shows the results:
the MCI value, the $J_\delta$ per radius, its plot and the video of the closings.

1. Run **Setup** and **Helpers** once.
2. Use one of the three options to load your dual cloud:
   - **Option 1**: one CSV file with the columns `x`, `y`, `Type`.
   - **Option 2**: two NumPy arrays.
   - **Option 3**: two files, one per point set (`.csv` or `.npy`).
3. Optionally get a recommended radius range, then run the MCI.

Each example below already points to data of the project, so the notebook can be run as it is.
The results of each pair are saved in `notebook/outputs/<name>/`.

## Setup

In [ ]:
import sys
import shutil
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Video, Markdown

#Project root: this notebook lives in <project>/notebook
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from mci_lib.mci.core import MCITestFamily
from mci_lib.nearest_neighbour_metrics.nn_distance import calculate_nearest_neighbour_distance_distribution_from_file
from mci_lib.nearest_neighbour_metrics.nn_distance_summarizer import summarize_nn_distances_from_folder, offer_radius_ranges_recommendation

#The results of each dual cloud are saved in notebook/outputs/<name>/
OUTPUT_DIR = PROJECT_ROOT / "notebook" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

#Default radius range of the MCI, each run can use its own (see recommend_radius_range)
MIN_RADIUS, MAX_RADIUS, RADIUS_STEP = 8, 32, 4

## Helpers

In [ ]:
def load_dual_cloud_csv(path, label_1="Type 1", label_2="Type 2"):
    '''Loads a dual cloud from one csv with the columns x, y and Type (the format used by the pipeline).'''
    df = pd.read_csv(path)
    A = df[df["Type"] == label_1][["x", "y"]].to_numpy(dtype=float)
    B = df[df["Type"] == label_2][["x", "y"]].to_numpy(dtype=float)
    return A, B


def load_point_set(path, label=None):
    '''Loads one point set from a .npy file (N x 2 array) or a csv with the columns x and y.
    If the csv has a Type column, only the rows of label are loaded.'''
    path = Path(path)
    if path.suffix == ".npy":
        return np.load(path).astype(float)[:, :2]
    df = pd.read_csv(path)
    if label is not None and "Type" in df.columns:
        df = df[df["Type"] == label]
    return df[["x", "y"]].to_numpy(dtype=float)


def preview(A, B, name):
    '''Plots both point sets: the first in red, the second in cyan.'''
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(A[:, 0], A[:, 1], ".r", markersize=3, label=f"Point set 1 ({len(A)} points)")
    ax.plot(B[:, 0], B[:, 1], ".c", markersize=3, label=f"Point set 2 ({len(B)} points)")
    ax.set_aspect("equal")
    ax.set_title(name)
    ax.legend(loc="lower left", bbox_to_anchor=(0, 1.05))
    plt.show()


def recommend_radius_range(A, B):
    '''Prints the radius range recommended from the nearest neighbour distances, as step 4 of the pipeline does.'''
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        cloud = pd.concat([pd.DataFrame({"x": A[:, 0], "y": A[:, 1], "Type": "Type 1"}),
                           pd.DataFrame({"x": B[:, 0], "y": B[:, 1], "Type": "Type 2"})])
        cloud.to_csv(tmp / "cloud.csv", index=False)
        (tmp / "nn").mkdir()
        calculate_nearest_neighbour_distance_distribution_from_file(tmp / "cloud.csv", tmp / "nn")
        summarize_nn_distances_from_folder(tmp / "nn", tmp / "summary", "stats.csv")
        offer_radius_ranges_recommendation(tmp / "summary", "stats.csv")


def show_results(name):
    '''Shows the MCI, the J_delta per radius, its plot and the video of a dual cloud already computed.'''
    folder = OUTPUT_DIR / name
    measurements = pd.read_csv(folder / "measurements.csv")
    mci = measurements["mci"].iloc[0]
    jd_r = [pair.strip("()").split(",") for pair in measurements["jd_r"].iloc[0].split(";")]
    jd_table = pd.DataFrame(jd_r, columns=["radius", "J_delta"]).astype(float)

    display(Markdown(f"### MCI of `{name}`: **{mci:.4f}**"))
    display(jd_table.T)
    display(Image(filename=str(folder / name / f"{name}_jd_per_r_plot.png"), width=500))
    display(Video(str(folder / name / f"{name}.mp4"), embed=True, width=600))
    return mci


def run_mci(A, B, name, min_radius=MIN_RADIUS, max_radius=MAX_RADIUS, radius_step=RADIUS_STEP):
    '''Computes the MCI of the dual cloud as mci_process.py does, saves it in outputs/<name>/ and shows the results.
    name is used as folder and file name, so use only letters, numbers, - and _.'''
    radius_list = list(range(min_radius, max_radius + radius_step, radius_step))
    #measurements.csv is appended on each run, so a rerun with the same name starts from an empty folder
    shutil.rmtree(OUTPUT_DIR / name, ignore_errors=True)
    #metadata_override is needed for points that are not randomly generated, otherwise the name is parsed as seed/translation/rotation
    metadata = {"name": name, "points_1": len(A), "points_2": len(B)}
    test = MCITestFamily(radius_list=radius_list, scale_factor=1,
                         A_points=np.asarray(A, dtype=float), B_points=np.asarray(B, dtype=float),
                         test_name=name, metadata_override=metadata, folder_name_override=name)
    test.generate_mci_files("measurements", output_folder_name=str(OUTPUT_DIR))
    return show_results(name)

## Option 1: one CSV file with `x`, `y`, `Type`

The file has one row per point: its coordinates `x`, `y` and the point set it belongs to in `Type`
(`Type 1` for the first set, `Type 2` for the second). This is the format of every point set file of the pipeline.
Use `label_1` and `label_2` if your file uses other names.

The example is a perturbed Perlin cloud of the project (ε = 40). With radius 8–32 it gives MCI **0.6434**,
the same value as the pipeline.

In [ ]:
CSV_FILE = PROJECT_ROOT / "input_datasets" / "perturbation" / "B4816-1-A_Per_S_700003_E_40.0.csv"
NAME = "example_xytype"

A, B = load_dual_cloud_csv(CSV_FILE)
preview(A, B, NAME)

In [ ]:
#Optional: radius range recommended for this dual cloud
recommend_radius_range(A, B)

In [ ]:
run_mci(A, B, NAME, min_radius=8, max_radius=32, radius_step=4)

## Option 2: two NumPy arrays

`A` and `B` are arrays of shape (N, 2) with the `x`, `y` coordinates of each point.
They can have a different number of points.

The example creates 500 random points and moves each one a little to get the second set.

In [ ]:
rng = np.random.default_rng(0)
A = rng.uniform(0, 1000, size=(500, 2))
B = A + rng.normal(0, 5, size=A.shape)
NAME = "example_numpy"

preview(A, B, NAME)

In [ ]:
#Optional: radius range recommended for this dual cloud
recommend_radius_range(A, B)

In [ ]:
#Radius range recommended above for this example
run_mci(A, B, NAME, min_radius=16, max_radius=64, radius_step=8)

## Option 3: two files, one per point set

Each file holds one point set:
- a `.csv` with the columns `x` and `y` (if it also has a `Type` column, pass `label` to choose the rows), or
- a `.npy` file with an (N, 2) array, as saved by `np.save`.

The example compares two different base Perlin clouds of the project (S4-1-A and S4-1-B).
With radius 8–24 it gives MCI **0.55**, the value of that pair in the cross heatmap.

In [ ]:
FILE_1 = PROJECT_ROOT / "input_datasets" / "base" / "S4-1-A_T(0,0)_R0_.csv"
FILE_2 = PROJECT_ROOT / "input_datasets" / "base" / "S4-1-B_T(0,0)_R0_.csv"
NAME = "example_two_files"

A = load_point_set(FILE_1, label="Type 1")
B = load_point_set(FILE_2, label="Type 1")
preview(A, B, NAME)

In [ ]:
#Optional: radius range recommended for this dual cloud
recommend_radius_range(A, B)

In [ ]:
run_mci(A, B, NAME, min_radius=8, max_radius=24, radius_step=4)

## See a result again

Every computed dual cloud stays in `notebook/outputs/<name>/`:
`measurements.csv`, and in `<name>/` the plot `<name>_jd_per_r_plot.png`, the video `<name>.mp4`
and the frames of the video. Show any of them again without recomputing it:

In [ ]:
show_results("example_xytype")